# Transformer from scratch



 1. Implementation of Encoder, Decoder and Transformer blocks
 2. Application in translation and training on a small dataset

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math



# Multihead Attention Block

The heart of transformer lies in Attention Mechanism, which is a scaled dot product of three vectors (Query,Key,Values).
- Query → It is a vector representing current word, for which we want to calculate the attention weights.
- Key → It is a vector which acts as an identifier, that helps to determine if a part of the sequence is relevant to what the query is looking for.
- Value → It is a vector which carry the actual information that will be used to build the next layer’s representation.

A multihead attention consists of h blocks of attention stacked up in parallel.

Self-attention, often called intra-attention, allows each word in a sequence to focus on different words in the same sequence, helping the model understand relationships and dependencies between words.

In [4]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        
        assert d_model % num_heads == 0, "d_model doit être divisible par num_heads"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads  # Dimension par tête
        
        #TODO: Créez les projections linéaires pour Q, K, V et la sortie
        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        self.W_out = nn.Linear(d_model, d_model)

    def scaled_dot_product_attention(self, q, k, v, mask=None):
        """
        Calcule l’attention : softmax((Q.K^T) / sqrt(d_k)) * V
        Entrées: q, k, v de forme (B, H, L, d_k), mask optionnel
        """
        d_k = q.size(-1)
        
        # TODO: Calculez les scores de similarité
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(d_k)
        
        # TODO: Appliquez un masque si nécessaire
        if mask is not None:
          scores = scores.masked_fill(mask == 0, -1e9)
        
        # TODO: Appliquez softmax pour obtenir les poids d'attention
        weights = F.softmax(scores, dim=-1)
        
        # TODO: Multipliez les poids par V
        attention = torch.matmul(weights, v)
        
        return attention

    def forward(self, q, k, v, mask=None):
        """
        Compute multi-head attention.
        q, k, v: input sequences of shape (batch_size, seq_len, d_model)
        mask: optional mask tensor
        """
        batch_size = q.size(0)
        
        # 1.Appliquez les projections linéaires
        q = self.W_Q(q)
        k = self.W_K(k)
        v = self.W_V(v)
        
        # 2.  Redimensionnez pour séparer les têtes : (B, L, d_model) -> (B, H, L, d_k)
        q = q.view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        k = k.view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        v = v.view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        
        # 3. Calculez l'attention multi-tête
        attention = self.scaled_dot_product_attention(q, k, v, mask)
        # Shape: (B, num_heads, seq_len, d_k)
        
        # 4. Concaténez les têtes et projetez en sortie: (B, num_heads, L, d_k) -> (B, L, d_model)
        attention = attention.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model) # to do
        
        # 5. Final linear projection
        output = self.W_out(attention)  # Shape: (B, L, d_model)
        
        return output


# Valider votre implementation

In [5]:
# Test rapide
attention = MultiHeadAttention(d_model=512, num_heads=8)
x = torch.rand(2, 10, 512)  # (batch_size, seq_length, d_model)
out = attention(x, x, x)

print("Output shape:", out.shape)  # Doit être (2, 10, 512)

Output shape: torch.Size([2, 10, 512])


# Position Wise Feed Forward
The attention sublayer in encoder and decoder is followed by a fully connected feed-forward layer, which is applied to each position separately and identically, it consists of two linear transformation with a ReLU activation in between.

In [6]:
class PositionWiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1) -> None:
        super(PositionWiseFeedForward, self).__init__()

        # First linear transformation (increases dimension)
        self.linear1 = nn.Linear(d_model, d_ff)

        # Second linear transformation (restores dimension)
        self.linear2 = nn.Linear(d_ff, d_model)

        # Dropout layer to prevent overfitting
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        """
        x: shape (batch_size, seq_len, d_model)

        Applies:
        - Linear1 (d_model → d_ff)
        - ReLU activation
        - Dropout
        - Linear2 (d_ff → d_model)
        """
        # Transformation linéaire + ReLU
        x = self.linear1(x)
        x = torch.relu(x)

        # Dropout
        x = self.dropout(x)

        # Transformation linéaire de retour
        x = self.linear2(x)
        
        return x

In [7]:
x = torch.rand(2, 5, 16)  # batch_size=2, sequence length=5, vector dim=16
ff = PositionWiseFeedForward(d_model=16, d_ff=64)
output = ff(x)
print("Output shape:", output.shape)


Output shape: torch.Size([2, 5, 16])


# Encoder block

It is composed of 6 identical layers , each layer has two sublayers:

- subLayer1 → Multi-Head Self Attention followed by Add & norm
- subLayer2 → Feed Forward followed by Add & norm

Layer Normalization:

The multihead attention layer and Feed forward layer is followed by a add and normalization layer in both encoder and decoder.
The add operation is for residual connection(carries the previous information), which adds the input to the output of multihead attention, which is then normalized.

Position Embedding:

Transformers treat input sequence as set rather than a ordered sequence.

Positional embedding is added to the input embedding to give the model, information about the position of token in Sequence.


In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1) -> None:
        super(EncoderLayer, self).__init__()

        # Multi-head self-attention mechanism
        self.self_attn = MultiHeadAttention(d_model=d_model, num_heads=num_heads)

        # Position-wise feedforward network
        self.feed_forward = PositionWiseFeedForward(d_model, d_ff, dropout)

        # Layer normalization applied after residual connections
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

        # Dropout layers to regularize the network
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        """
        x: shape (batch_size, seq_len, d_model)
        mask: optional attention mask

        Steps:
        1. Apply multi-head self-attention on x
        2. Add residual connection and apply LayerNorm
        3. Apply position-wise feedforward network
        4. Add another residual connection and apply LayerNorm
        """

        # --- Multi-head self-attention ---
        attn_output = self.self_attn(x, x, x, mask)  # Q=K=V=x for self-attention
        x = x + self.dropout1(attn_output)           # Residual connection
        x = self.norm1(x)                            # Normalization

        # --- Feedforward network ---
        ff_output = self.feed_forward(x)
        x = x + self.dropout2(ff_output)             # Residual connection
        x = self.norm2(x)                            # Normalization

        return x

In [9]:
class Encoder(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, num_layers, input_vocab_size, max_len=512, dropout=0.1):
        super(Encoder, self).__init__()

        # Embedding layer to convert token IDs into dense vectors
        self.embedding = nn.Embedding(input_vocab_size, d_model)

        # Fixed positional encoding matrix (not learned)
        self.positional_encoding = nn.Parameter(self._get_positional_encoding(max_len, d_model), requires_grad=False)

        # Stack of N Encoder Layers
        self.layers = nn.ModuleList([
            EncoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)
        ])

        # Dropout applied to the input embeddings
        self.dropout = nn.Dropout(dropout)

    def _get_positional_encoding(self, max_len, d_model):
        """
        Create sinusoidal positional encodings of shape (1, max_len, d_model)
        This allows the model to take order of tokens into account.
        """
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)  # Apply sin to even indices
        pe[:, 1::2] = torch.cos(position * div_term) # Apply cos to odd indices

        return pe.unsqueeze(0)  # shape: (1, max_len, d_model)

    def forward(self, x, mask=None):
        """
        x: input token indices (batch_size, seq_len)
        mask: optional attention mask

        1. Embed the tokens
        2. Add positional encoding
        3. Apply dropout
        4. Pass through N encoder layers
        """
        # Step 1–2: Embed + add positional encodings
        x = self.embedding(x) + self.positional_encoding[:, :x.size(1), :]

        # Step 3: Apply dropout to input
        x = self.dropout(x)

        # Step 4: Pass through the stack of encoder layers
        for layer in self.layers:
            x = layer(x, mask)

        return x  # Final encoded representation of the input

# Decoder Block

It is composed of 6 identical layers , each layer has two sublayers:

- sublayer1 → Multi-Head Self Attention followed by Add & Norm
- sublayer2 → Multi-Head Self Attention Over Encoder’s Output followed by Add & Norm
- sublayer3 → Feed Forward followed by Add & Norm

In [11]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super(DecoderLayer, self).__init__()

        # TODO: Define masked self-attention and cross-attention
        self.self_attn = MultiHeadAttention(d_model=d_model, num_heads=num_heads)
        self.cross_attn = MultiHeadAttention(d_model=d_model, num_heads=num_heads)

        # TODO: Define feed-forward network
        self.feed_forward = PositionWiseFeedForward(d_model, d_ff, dropout)

        # TODO: Add LayerNorm for each sub-layer
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)

        # TODO: Add Dropout layers
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)

    def forward(self, x, enc_output, src_mask=None, tgt_mask=None):
        # Step 1: Masked self-attention
        attn_output = self.self_attn(x, x, x, tgt_mask)
        x = x + self.dropout1(attn_output)
        x = self.norm1(x)

        # Step 2: Cross-attention (decoder attends to encoder)
        attn_output = self.cross_attn(x, enc_output, enc_output, src_mask)
        x = x + self.dropout2(attn_output)
        x = self.norm2(x)

        # Step 3: Feed-forward
        ff_output = self.feed_forward(x)
        x = x + self.dropout3(ff_output)
        x = self.norm3(x)

        return x


In [27]:
class Decoder(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, num_layers, output_vocab_size, max_len=512, dropout=0.1):
        super(Decoder, self).__init__()
        # Token embedding layer to map output tokens to vector representations
        self.embedding = nn.Embedding(output_vocab_size, d_model)

        # Fixed positional encoding to inject token order information
        self.positional_encoding = nn.Parameter(self._get_positional_encoding(max_len, d_model), requires_grad=False)

        # Stack of decoder layers
        self.layers = nn.ModuleList([
            DecoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)
        ])

        # Dropout for regularization after input embedding
        self.dropout = nn.Dropout(dropout)

        # Final linear layer to project decoder output to vocabulary size
        self.fc_out = nn.Linear(d_model, output_vocab_size)

    def _get_positional_encoding(self, max_len, d_model):
        """
        Create sinusoidal positional encoding matrix of shape (1, max_len, d_model)
        to encode the position of each token in the sequence.
        """
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

        # Even positions: sin(pos / (10000^(2i/d_model)))
        pe[:, 0::2] = torch.sin(position * div_term)

        # Odd positions: cos(pos / (10000^(2i/d_model)))
        pe[:, 1::2] = torch.cos(position * div_term)

        return pe.unsqueeze(0)  # Shape: (1, max_len, d_model)

    def forward(self, x, enc_output, src_mask=None, tgt_mask=None):
        """
        x: target input tokens (batch_size, tgt_seq_len)
        enc_output: encoder output (batch_size, src_seq_len, d_model)
        src_mask: source mask (optional)
        tgt_mask: target mask (to prevent attending to future positions)
        """
        # Step 1: Add embeddings and positional encoding
        x = self.embedding(x) + self.positional_encoding[:, :x.size(1), :]
        x = self.dropout(x)

        # Step 2: Pass through each decoder layer
        for layer in self.layers:
            x = layer(x, enc_output, src_mask, tgt_mask)

        # Step 3: Map output to vocabulary space
        x = self.fc_out(x)
        return x

# Transformer Block

Now let’s combine it all together, stacking up the encoder and decoder blocks forms a transformer. The input to the transformer is source and target vectors, which are used to form the src_mask and trg_mask .

Source masks in the encoder prevents attention to padding tokens, while target masks in the decoder blocks future tokens (look-ahead masking) to ensure autoregressive generation.

The source vector and source mask are passed as input to encoder block, to obtain the encoder’s output, which is passed along with target vector, source mask and target mask to decoder, which gives the final output.

In [24]:
class Transformer(nn.Module):
    def __init__(self, embed_dim, src_vocab_size, target_vocab_size,
                 num_layers=6, d_ff=2048, n_heads=8):
        super(Transformer, self).__init__()

        # Store vocab size for output projection (optional)
        self.target_vocab_size = target_vocab_size

        # Encoder: transforms source input into contextual representations
        self.encoder = Encoder(
            d_model=embed_dim,
            num_heads=n_heads,
            d_ff=d_ff,
            num_layers=num_layers,
            input_vocab_size=src_vocab_size,
            max_len=max_len
        )

        # Decoder: generates the output sequence by attending to encoder outputs
        self.decoder = Decoder(
            d_model=embed_dim,
            num_heads=n_heads,
            d_ff=d_ff,
            num_layers=num_layers,
            output_vocab_size=target_vocab_size,
            max_len=max_len
        )

        # Number of attention heads (can be useful for future use or diagnostics)
        self.num_heads = n_heads

    def generate_mask(self, src, trg):
        """
        Create masks for source and target sequences:
        - src_mask: prevents attention to padding tokens in the source
        - trg_mask: prevents attending to future tokens in the target during training
        """

        # Source mask: 1 where src is not padding (assume padding token is 0)
        src_mask = (src != 0).unsqueeze(1).unsqueeze(2)  
        # Shape: (batch_size, 1, 1, src_len)

        # Target mask: lower triangular matrix to prevent attending to future tokens
        batch_size, trg_len = trg.size()
        trg_mask = torch.tril(torch.ones((trg_len, trg_len), device=trg.device)).bool()
        trg_mask = trg_mask.unsqueeze(0).unsqueeze(1)
        # Shape: (batch_size, 1, trg_len, trg_len)

        return src_mask, trg_mask

    def forward(self, src, trg):
        """
        src: source input tokens (batch_size, src_seq_len)
        trg: target input tokens (batch_size, trg_seq_len)
        """

        # Step 1: Create masks for padding and future tokens
        src_mask, trg_mask = self.generate_mask(src, trg)

        # Step 2: Encode the source sequence
        enc_out = self.encoder(src, src_mask) # Shape: (batch_size, src_len, embed_dim)

        # Step 3: Decode the target sequence using encoder outputs as context
        outputs = self.decoder(trg, enc_out, src_mask, trg_mask)

        return outputs

# Example with training  on a small subset

We will be training the transformer which we implemented for language translation task(English to French). We will take a very small dataset and vocabulary so that we can overfit the model on these sample translation sentences and check its performance if its working for the examples which we trained on.


## Dataset Preparation
The dataset class takes index as input, for that particular index it converts the sentences into its token using the source and target vocabularies. In case if a word is not present it replaces it with <unk> token. The tokens are finally padded as per the max_len .

In [17]:
import torch
from torch.utils.data import DataLoader, Dataset

class TranslationDataset(Dataset):
    def __init__(self, source_sentences, target_sentences, src_vocab, trg_vocab, max_len):
        self.source_sentences = source_sentences
        self.target_sentences = target_sentences
        self.src_vocab = src_vocab
        self.trg_vocab = trg_vocab
        self.max_len = max_len

    def __len__(self):
        return len(self.source_sentences)

    def pad_sequence(self, seq, vocab):
        # Add <eos> token and pad the sequence to max_len
        seq = seq + [vocab["<eos>"]]
        if len(seq) < self.max_len:
            seq += [vocab["<pad>"]] * (self.max_len - len(seq))
        else:
            seq = seq[:self.max_len]
        return seq

    def __getitem__(self, idx):
        src = self.source_sentences[idx]
        trg = self.target_sentences[idx]
        # Convert words to indexes
        src_indexes = [self.src_vocab.get(word, self.src_vocab["<unk>"]) for word in src]
        trg_indexes = [self.trg_vocab.get(word, self.trg_vocab["<unk>"]) for word in trg]
        # Pad sequences
        src_indexes = self.pad_sequence(src_indexes, self.src_vocab)
        trg_indexes = self.pad_sequence(trg_indexes, self.trg_vocab)
        return torch.tensor(src_indexes), torch.tensor(trg_indexes)

source_sentences = [
    ["hello", "world"],
    ["i", "am", "learning", "transformers"],
    ["pytorch", "is", "great"],
    ["how", "are", "you"],
    ["openai", "makes", "amazing", "models"],
    ["i", "love", "coding"],
    ["you", "are", "awesome"],
    ["transformers", "are", "powerful"],
    ["machine", "learning", "is", "fun"],
    ["let's", "build", "a", "translator"],
    ["artificial", "intelligence", "is", "fascinating"],
    ["deep", "learning", "is", "a", "subset", "of", "machine", "learning"],
    ["the", "cat", "is", "on", "the", "mat"],
    ["this", "is", "a", "beautiful", "day"],
    ["natural", "language", "processing", "is", "interesting"],
    ["i", "enjoy", "solving", "problems"],
    ["technology", "is", "evolving", "rapidly"],
    ["data", "science", "is", "a", "growing", "field"],
    ["the", "sun", "rises", "in", "the", "east"],
    ["the", "moon", "is", "bright", "tonight"],
    ["we", "are", "exploring", "the", "universe"],
    ["this", "task", "requires", "patience"],
    ["the", "quick", "brown", "fox", "jumps", "over", "the", "lazy", "dog"],
    ["quantum", "computing", "is", "the", "future"],
    ["computer", "vision", "is", "a", "field", "of", "AI"]
]

target_sentences = [
    ["bonjour", "le", "monde"],
    ["j'apprends", "les", "transformers"],
    ["pytorch", "est", "génial"],
    ["comment", "vas-tu"],
    ["openai", "crée", "des", "modèles", "incroyables"],
    ["j'adore", "coder"],
    ["tu", "es", "génial"],
    ["les", "transformers", "sont", "puissants"],
    ["l'apprentissage", "automatique", "est", "amusant"],
    ["construisons", "un", "traducteur"],
    ["l'intelligence", "artificielle", "est", "fascinante"],
    ["le", "deep", "learning", "est", "un", "sous-ensemble", "de", "l'apprentissage", "automatique"],
    ["le", "chat", "est", "sur", "le", "tapis"],
    ["c'est", "une", "belle", "journée"],
    ["le", "traitement", "du", "langage", "naturel", "est", "intéressant"],
    ["j'aime", "résoudre", "des", "problèmes"],
    ["la", "technologie", "évolue", "rapidement"],
    ["la", "science", "des", "données", "est", "un", "domaine", "en", "croissance"],
    ["le", "soleil", "se", "lève", "à", "l'est"],
    ["la", "lune", "est", "brillante", "ce", "soir"],
    ["nous", "explorons", "l'univers"],
    ["cette", "tâche", "demande", "de", "la", "patience"],
    ["le", "renard", "brun", "rapide", "saute", "par-dessus", "le", "chien", "paresseux"],
    ["l'informatique", "quantique", "est", "l'avenir"],
    ["la", "vision", "par", "ordinateur", "est", "un", "domaine", "de", "l'IA"]
]

# Vocabularies (For simplicity, we're using small vocabularies here)
src_vocab = {
    "<sos>": 0, "<eos>": 1, "<pad>": 2, "<unk>": 3,
    "hello": 4, "world": 5, "i": 6, "am": 7, "learning": 8,
    "transformers": 9, "pytorch": 10, "is": 11, "great": 12,
    "how": 13, "are": 14, "you": 15, "openai": 16, "makes": 17,
    "amazing": 18, "models": 19, "love": 20, "coding": 21,
    "awesome": 22, "powerful": 23, "machine": 24, "fun": 25,
    "let's": 26, "build": 27, "a": 28, "translator": 29,
    "artificial": 30, "intelligence": 31, "fascinating": 32,
    "deep": 33, "subset": 34, "of": 35, "the": 36, "cat": 37,
    "on": 38, "mat": 39, "this": 40, "beautiful": 41, "day": 42,
    "natural": 43, "language": 44, "processing": 45, "interesting": 46,
    "enjoy": 47, "solving": 48, "problems": 49, "technology": 50,
    "evolving": 51, "rapidly": 52, "data": 53, "science": 54,
    "growing": 55, "field": 56, "sun": 57, "rises": 58, "in": 59,
    "east": 60, "moon": 61, "bright": 62, "tonight": 63, "we": 64,
    "exploring": 65, "universe": 66, "task": 67, "requires": 68,
    "patience": 69, "quick": 70, "brown": 71, "fox": 72, "jumps": 73,
    "over": 74, "lazy": 75, "dog": 76, "quantum": 77, "computing": 78,
    "future": 79, "computer": 80, "vision": 81, "ai": 82
}

trg_vocab = {
    "<sos>": 0, "<eos>": 1, "<pad>": 2, "<unk>": 3,
    "bonjour": 4, "le": 5, "monde": 6, "j'apprends": 7, "les": 8,
    "transformers": 9, "pytorch": 10, "est": 11, "génial": 12,
    "comment": 13, "vas-tu": 14, "openai": 15, "crée": 16,
    "des": 17, "modèles": 18, "incroyables": 19, "j'adore": 20,
    "coder": 21, "tu": 22, "es": 23, "génial": 24,  # 'génial' doublé → fusionné
    "sont": 25, "puissants": 26, "l'apprentissage": 27,
    "automatique": 28, "amusant": 29, "construisons": 30,
    "un": 31, "traducteur": 32, "l'intelligence": 33,
    "artificielle": 34, "fascinante": 35, "deep": 36,
    "learning": 37, "sous-ensemble": 38, "de": 39,
    "chat": 40, "sur": 41, "tapis": 42, "c'est": 43,
    "une": 44, "belle": 45, "journée": 46, "traitement": 47,
    "du": 48, "langage": 49, "naturel": 50, "intéressant": 51,
    "j'aime": 52, "résoudre": 53, "problèmes": 54,
    "la": 55, "technologie": 56, "évolue": 57, "rapidement": 58,
    "science": 59, "données": 60, "domaine": 61, "en": 62,
    "croissance": 63, "soleil": 64, "se": 65, "lève": 66,
    "à": 67, "l'est": 68, "lune": 69, "brillante": 70,
    "ce": 71, "soir": 72, "nous": 73, "explorons": 74,
    "l'univers": 75, "cette": 76, "tâche": 77, "demande": 78,
    "patience": 79, "renard": 80, "brun": 81, "rapide": 82,
    "saute": 83, "par-dessus": 84, "chien": 85, "paresseux": 86,
    "l'informatique": 87, "quantique": 88, "avenir": 89,
    "vision": 90, "par": 91, "ordinateur": 92, "de": 93,
    "l'IA": 94
}



# Maximum length for padding
max_len = 10

# Create dataset
dataset = TranslationDataset(source_sentences, target_sentences, src_vocab, trg_vocab, max_len)
dataloader = DataLoader(dataset, batch_size=4, shuffle=True)


# Training code

Now let’s train the model on our sample dataset. Note that we are trying to overfit the model, just to check its working.

In [28]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

# Define the training loop
def train(model, dataloader, criterion, optimizer, num_epochs, trg_pad_idx):
    model.train()
    for epoch in range(num_epochs):
        epoch_loss = 0
        for batch_idx, (src, trg) in enumerate(dataloader):
            src, trg = src, trg
            # Remove <eos> token from target to get trg_input
            trg_input = trg[:, :-1]
            trg_output = trg[:, 1:]
            # Get predictions
            preds = model(src, trg_input)
            # Reshape predictions and targets for loss calculation
            preds = preds.reshape(-1, preds.shape[-1])
            trg_output = trg_output.reshape(-1)
            # Calculate loss
            loss = criterion(preds, trg_output)
            # Backpropagation and optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss / len(dataloader):.4f}')

# Hyperparameters and setup
embed_dim = 512
src_vocab_size = max(src_vocab.values())+1
target_vocab_size = max(trg_vocab.values())+1
num_layers = 2
n_heads = 8
learning_rate = 0.0001
num_epochs = 100
trg_pad_idx = trg_vocab["<pad>"]
# Initialize model, criterion, and optimizer
model = Transformer(embed_dim, src_vocab_size, target_vocab_size, num_layers, 2048, n_heads)

criterion = nn.CrossEntropyLoss(ignore_index=trg_pad_idx)
optimizer = optim.Adam(model.parameters(), lr=learning_rate)
# Initialize DataLoader
batch_size = 4
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
# Start training
train(model, dataloader, criterion, optimizer, num_epochs, trg_pad_idx)


Epoch [1/100], Loss: 4.1535
Epoch [2/100], Loss: 3.4139
Epoch [3/100], Loss: 2.7516
Epoch [4/100], Loss: 2.2542
Epoch [5/100], Loss: 1.7327
Epoch [6/100], Loss: 1.2790
Epoch [7/100], Loss: 0.8758
Epoch [8/100], Loss: 0.7434
Epoch [9/100], Loss: 0.4898
Epoch [10/100], Loss: 0.4306
Epoch [11/100], Loss: 0.3236
Epoch [12/100], Loss: 0.2744
Epoch [13/100], Loss: 0.1979
Epoch [14/100], Loss: 0.1561
Epoch [15/100], Loss: 0.1509
Epoch [16/100], Loss: 0.1382
Epoch [17/100], Loss: 0.0945
Epoch [18/100], Loss: 0.0945
Epoch [19/100], Loss: 0.0842
Epoch [20/100], Loss: 0.0751
Epoch [21/100], Loss: 0.0635
Epoch [22/100], Loss: 0.0632
Epoch [23/100], Loss: 0.0596
Epoch [24/100], Loss: 0.0539
Epoch [25/100], Loss: 0.0450
Epoch [26/100], Loss: 0.0400
Epoch [27/100], Loss: 0.0397
Epoch [28/100], Loss: 0.0406
Epoch [29/100], Loss: 0.0385
Epoch [30/100], Loss: 0.0352
Epoch [31/100], Loss: 0.0343
Epoch [32/100], Loss: 0.0318
Epoch [33/100], Loss: 0.0289
Epoch [34/100], Loss: 0.0287
Epoch [35/100], Loss: 0

## Inference

We can now test the trained model on some sample sentences.

In [29]:
import torch

def predict(model, src_sentence, src_vocab, trg_vocab, max_len=50):
    model.eval()  # Set the model to evaluation mode
    # Tokenize and convert the src_sentence to a tensor
    src_indexes = [src_vocab[token] for token in src_sentence.split()]
    src_tensor = torch.LongTensor(src_indexes).unsqueeze(0)  # Add batch dimension
    # Initialize the target sequence with the start token
    trg_indexes = [trg_vocab["<sos>"]]
    # Loop to generate each word in the target sequence
    for i in range(max_len):
        trg_tensor = torch.LongTensor(trg_indexes).unsqueeze(0)  # Add batch dimension
        with torch.no_grad():
            preds = model(src_tensor, trg_tensor)
        # Get the index of the highest probability token
        pred_token = preds.argmax(2)[:, -1].item()
        trg_indexes.append(pred_token)
        # Stop if the model predicts the end of sentence token
        if pred_token == trg_vocab["<eos>"]:
            break
    # Convert the predicted indexes back to words
    trg_tokens = [list(trg_vocab.keys())[list(trg_vocab.values()).index(i)] for i in trg_indexes]
    return trg_tokens[1:]  # Return the prediction excluding the start token

src_sentence = " moon is bright"
predicted_sentence = predict(model, src_sentence, src_vocab, trg_vocab)
print("Predicted Translation:", " ".join(predicted_sentence))

Predicted Translation: lune est brillante ce soir soir soir soir <eos>


In [42]:
src_sentence = "patience"
predicted_sentence = predict(model, src_sentence, src_vocab, trg_vocab)
print("Predicted Translation:", " ".join(predicted_sentence))

Predicted Translation: de la patience le chien paresseux <eos>


In [43]:
src_sentence = "i am coding"
predicted_sentence = predict(model, src_sentence, src_vocab, trg_vocab)
print("Predicted Translation:", " ".join(predicted_sentence))

Predicted Translation: résoudre des problèmes <eos>


In [44]:
src_sentence = "i am learning"
predicted_sentence = predict(model, src_sentence, src_vocab, trg_vocab)
print("Predicted Translation:", " ".join(predicted_sentence))

Predicted Translation: les transformers sont puissants <eos>
